In [4]:
import pandas as pd

df = pd.read_csv(r"C:\Users\hp\Downloads\archive (1)\crime_incidents_messy.csv")

df.head()

,incident_id,crime_type,district,city,state,address,latitude,longitude,incident_datetime,officer_id,...,victim_gender,victim_phone,weapon_used,severity,case_status,resolution,num_arrests,property_loss_usd,reported_online,notes
0,INC001115,asslt,Sou,Maplewood,GA,2830 Cedar Lane,170.284100,-77.500710,2024-04-16 08:45:03,OFF0078,...,Unknown,6223265920,Firearm,2,Open,No Arrest,NaN,NaN,True,Incident at 2830 Cedar Lane. Officer responded...
1,INC004706,burglary,southeast,Maplewood,OH,2361 Park Rd,29.422717,-77.167016,2022-01-11 22:03:29,OFF0059,...,NaN,241-973-4826,Firearm,3,NaN,NaN,2.0,44839.49,yes,Incident at 2361 Park Rd. Officer responded at...
2,INC002249,Homocide,Southwest,Lakewood,AZ,1067 Main St,39.411460,-98.615298,2022-04-14 11:39:24,OFF0088,...,Other,244-584-7696,KNIFE,1,CLOSED,warning,2.0,15963.38,YES,Incident at 1067 Main St. Officer responded at...
3,INC000021,Property Damage,Cen,Springfield,AZ,4713 Washington Ave,28.633439,-99.030051,2020-12-09 15:14:24,OFF0077,...,NaN,5021227484,hands,Low,Resolved,NaN,5.0,48680.14,False,Incident at 4713 Washington Ave. Officer respo...
4,INC000488,Domestc Violence,North,Lakewood,PA,1371 River Rd,34.217850,-121.614731,2021-07-24 20:08:13,OFF0083,...,M,9698766873,Unarmed,MEDIUM,Closed,Warning Issued,2.0,23513.01,YES,NaN


In [5]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (5250, 33)

Column Names:
['incident_id', 'crime_type', 'district', 'city', 'state', 'address', 'latitude', 'longitude', 'incident_datetime', 'officer_id', 'officer_first_name', 'officer_last_name', 'badge_number', 'suspect_id', 'suspect_first_name', 'suspect_last_name', 'suspect_age', 'suspect_gender', 'suspect_race', 'victim_id', 'victim_first_name', 'victim_last_name', 'victim_age', 'victim_gender', 'victim_phone', 'weapon_used', 'severity', 'case_status', 'resolution', 'num_arrests', 'property_loss_usd', 'reported_online', 'notes']

Data Types:
incident_id               str
crime_type                str
district                  str
city                      str
state                     str
address                   str
latitude              float64
longitude             float64
incident_datetime         str
officer_id                str
officer_first_name        str
officer_last_name         str
badge_number          float64
suspect_id                str
suspect_fi

In [6]:
before_rows = len(df)
before_duplicates = df.duplicated().sum()
before_nulls = df.isnull().sum().sum()

print("Before Cleaning")
print("Rows:", before_rows)
print("Duplicate Rows:", before_duplicates)
print("Total Missing Values:", before_nulls)

Before Cleaning
Rows: 5250
Duplicate Rows: 200
Total Missing Values: 16478


In [7]:
missing_report = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_report = missing_report[missing_report['Missing Count'] > 0]

missing_report.sort_values('Missing Count', ascending=False)

,Missing Count,Missing %
suspect_race,1601,30.50
suspect_gender,1413,26.91
suspect_age,1097,20.90
notes,1069,20.36
victim_phone,1056,20.11
victim_gender,1000,19.05
weapon_used,970,18.48
resolution,951,18.11
suspect_last_name,810,15.43
suspect_first_name,810,15.43


## Missing Data Handling Strategy

Different strategies were used depending on the meaning and data type of each column:

- **Categorical columns:** Missing values were replaced with `"Unknown"` because the original category cannot be reliably inferred.
- **Name, ID, and phone fields:** Missing values were replaced with `"Unknown"` or `"Not Provided"` to preserve the incident records.
- **Age columns:** Missing ages were replaced with the median age because median is less affected by extreme age values.
- **Numerical columns:** Missing numerical values such as latitude, longitude, and number of arrests were replaced using the median.
- **Property loss:** Missing values will be handled after converting the column to numeric format.
- **Incident datetime:** Missing dates will be handled separately because date information is important for time-based analysis.
- **Notes:** Missing notes were replaced with `"No Notes"` because absence of a note does not make the incident record invalid.

These decisions were made to preserve as many valid incident records as possible while avoiding unnecessary row deletion.

In [8]:
# Convert numeric columns to numeric datatype
df['property_loss_usd'] = pd.to_numeric(df['property_loss_usd'], errors='coerce')

# Fill missing numerical values with median
numeric_columns = [
    'latitude',
    'longitude',
    'badge_number',
    'suspect_age',
    'victim_age',
    'num_arrests',
    'property_loss_usd'
]

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

print("Numeric missing values handled.")
print(df[numeric_columns].isnull().sum())


Numeric missing values handled.
latitude             0
longitude            0
badge_number         0
suspect_age          0
victim_age           0
num_arrests          0
property_loss_usd    0
dtype: int64


In [9]:
# Fill missing categorical/text values
categorical_columns = [
    'suspect_gender',
    'suspect_race',
    'victim_gender',
    'weapon_used',
    'severity',
    'case_status',
    'resolution',
    'reported_online'
]

for col in categorical_columns:
    df[col] = df[col].fillna('Unknown')

# Fill missing names, IDs and phone numbers
text_columns = [
    'suspect_id',
    'suspect_first_name',
    'suspect_last_name',
    'victim_first_name',
    'victim_last_name',
    'victim_phone'
]

for col in text_columns:
    df[col] = df[col].fillna('Not Provided')

# Fill missing notes
df['notes'] = df['notes'].fillna('No Notes')

print("Categorical and text missing values handled.")

Categorical and text missing values handled.


In [10]:
print("Total Missing Values After Handling:", df.isnull().sum().sum())

print("\nColumns with Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Total Missing Values After Handling: 597

Columns with Missing Values:
incident_datetime    340
victim_id            257
dtype: int64


In [11]:
# Handle missing victim IDs
df['victim_id'] = df['victim_id'].fillna('Not Provided')

# Convert incident_datetime to datetime
df['incident_datetime'] = pd.to_datetime(
    df['incident_datetime'],
    errors='coerce'
)

print("incident_datetime dtype:", df['incident_datetime'].dtype)
print("Missing victim_id:", df['victim_id'].isnull().sum())
print("Missing incident_datetime:", df['incident_datetime'].isnull().sum())

incident_datetime dtype: datetime64[us]
Missing victim_id: 0
Missing incident_datetime: 720


In [12]:
invalid_dates = df['incident_datetime'].isna().sum()

print("Missing/Invalid Dates:", invalid_dates)

df = df.dropna(subset=['incident_datetime'])

print("Rows after removing invalid dates:", len(df))
print("Remaining missing dates:", df['incident_datetime'].isna().sum())

Missing/Invalid Dates: 720
Rows after removing invalid dates: 4530
Remaining missing dates: 0


In [13]:
duplicates_before = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after removal:", df.duplicated().sum())
print("Rows after duplicate removal:", len(df))

Duplicate rows before removal: 173
Duplicate rows after removal: 0
Rows after duplicate removal: 4357


In [14]:
# Standardize text columns
text_columns = [
    'crime_type',
    'district',
    'city',
    'state',
    'weapon_used',
    'severity',
    'case_status',
    'resolution',
    'suspect_gender',
    'suspect_race',
    'victim_gender'
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.title()

print("Text standardization completed.")

Text standardization completed.


In [15]:
print("Crime Types:")
print(df['crime_type'].unique())

print("\nDistricts:")
print(df['district'].unique())

print("\nSuspect Gender:")
print(df['suspect_gender'].unique())

print("\nVictim Gender:")
print(df['victim_gender'].unique())

print("\nSeverity:")
print(df['severity'].unique())


Crime Types:
<StringArray>
[              'Asslt',            'Burglary',            'Homocide',
     'Property Damage',    'Domestc Violence',             'Robbery',
        'Manslaughter',                 'B&E',           'Narcotics',
             'Burglry',               'Theft',        'Drug Offense',
       'Armed Robbery',               'Arsen',           'Abduction',
            'Graffiti',            'Trespass',         'Cyber Crime',
         'Trespassing',      'Sexual Assault',               'Fraud',
       'Theft/Larceny',             'Larceny',               'Arson',
            'Homicide',        'Tresspassing',          'Cybercrime',
             'Hacking',             'Assault',      'Sexual Assualt',
        'Drug Offence',       'Drunk Driving',                'Duii',
              'Robbry',            'Vandlism',        'Fire Setting',
           'Vandalism',          'Kidnapping',              'D.U.I.',
             'Roberry',   'Domestic Violence',                '

In [16]:
# Standardize gender values
gender_map = {
    'M': 'Male',
    'F': 'Female'
}

df['suspect_gender'] = df['suspect_gender'].replace(gender_map)
df['victim_gender'] = df['victim_gender'].replace(gender_map)


# Standardize district abbreviations
district_map = {
    'Sou': 'South',
    'Cen': 'Central',
    'Eas': 'East',
    'Nor': 'North',
    'Wes': 'West',
    'Mid': 'Midtown'
}

df['district'] = df['district'].replace(district_map)


# Standardize severity values
severity_map = {
    '1': 'Low',
    '2': 'Medium',
    '3': 'High',
    '4': 'Critical',
    'Med': 'Medium',
    'Crit': 'Critical'
}

df['severity'] = df['severity'].replace(severity_map)

print("Gender, district and severity standardization completed.")


Gender, district and severity standardization completed.


In [17]:
crime_map = {
    'Asslt': 'Assault',
    'Assault': 'Assault',
    
    'Homocide': 'Homicide',
    'Homicide': 'Homicide',
    'Murder': 'Homicide',
    
    'Burglry': 'Burglary',
    'Burglary': 'Burglary',
    'B&E': 'Breaking & Entering',
    'Breaking & Entering': 'Breaking & Entering',
    
    'Domestc Violence': 'Domestic Violence',
    'Domestic Violence': 'Domestic Violence',
    'Dom. Violence': 'Domestic Violence',
    'Dv': 'Domestic Violence',
    
    'Robbry': 'Robbery',
    'Roberry': 'Robbery',
    'Robbery': 'Robbery',
    
    'Sex Assualt': 'Sexual Assault',
    'Sexual Assualt': 'Sexual Assault',
    'Sexual  Assualt': 'Sexual Assault',
    'Sexual  Assault': 'Sexual Assault',
    'Sex  Assault': 'Sexual Assault',
    'Sex Assault': 'Sexual Assault',
    'Sa': 'Sexual Assault',
    'Sexual Assault': 'Sexual Assault',
    
    'Tresspassing': 'Trespassing',
    'Trespass': 'Trespassing',
    'Trespassing': 'Trespassing',
    
    'Kidnaping': 'Kidnapping',
    'Kidnapping': 'Kidnapping',
    
    'Vandlism': 'Vandalism',
    'Vandalism': 'Vandalism',
    
    'Arsen': 'Arson',
    'Arson': 'Arson',
    'Fire Setting': 'Arson',
    
    'Drug Offence': 'Drug Offense',
    'Drug Offense': 'Drug Offense',
    'Drug  Offense': 'Drug Offense',
    'Drug  Offence': 'Drug Offense',
    'Drugs': 'Drug Offense',
    'Narcotics': 'Drug Offense',
    
    'Duii': 'DUI',
    'D.U.I.': 'DUI',
    'Dui': 'DUI',
    'Dwi': 'DUI',
    'Drunk Driving': 'DUI',
    
    'Cyber Crime': 'Cybercrime',
    'Cyber  Crime': 'Cybercrime',
    'Cybercrime': 'Cybercrime',
    'Hacking': 'Cybercrime',
    
    'Theft/Larceny': 'Theft',
    'Larceny': 'Theft',
    'Stealing': 'Theft',
    'Theft': 'Theft',
    
    'Armed Robbery': 'Armed Robbery',
    'Armed  Robbery': 'Armed Robbery',
    
    'Assault & Battery': 'Assault & Battery',
    'Assault  &  Battery': 'Assault & Battery',
    
    'Property Damage': 'Property Damage',
    'Property  Damage': 'Property Damage',
    
    'Fraud': 'Fraud',
    'Online Fraud': 'Fraud',
    'Scam': 'Fraud',
    'Deception': 'Fraud',
    'Fraudulent Activity': 'Fraud',
    
    'Abduction': 'Abduction',
    'Manslaughter': 'Manslaughter',
    'Graffiti': 'Graffiti',
    'Battery': 'Battery',
    'Hacking': 'Cybercrime'
}

df['crime_type'] = df['crime_type'].replace(crime_map)

print("Crime type standardization completed.")
print("\nUnique crime types after cleaning:")
print(df['crime_type'].unique())

Crime type standardization completed.

Unique crime types after cleaning:
<StringArray>
[            'Assault',            'Burglary',            'Homicide',
     'Property Damage',   'Domestic Violence',             'Robbery',
        'Manslaughter', 'Breaking & Entering',        'Drug Offense',
               'Theft',       'Armed Robbery',               'Arson',
           'Abduction',            'Graffiti',         'Trespassing',
          'Cybercrime',      'Sexual Assault',               'Fraud',
                 'DUI',           'Vandalism',          'Kidnapping',
             'Battery',   'Assault & Battery',  'Domestic  Violence']
Length: 24, dtype: str


In [18]:
df['crime_type'] = (
    df['crime_type']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print("Final unique crime types:", df['crime_type'].nunique())
print(df['crime_type'].unique())


Final unique crime types: 23
<StringArray>
[            'Assault',            'Burglary',            'Homicide',
     'Property Damage',   'Domestic Violence',             'Robbery',
        'Manslaughter', 'Breaking & Entering',        'Drug Offense',
               'Theft',       'Armed Robbery',               'Arson',
           'Abduction',            'Graffiti',         'Trespassing',
          'Cybercrime',      'Sexual Assault',               'Fraud',
                 'DUI',           'Vandalism',          'Kidnapping',
             'Battery',   'Assault & Battery']
Length: 23, dtype: str


In [19]:
# Detect outliers using IQR method

outlier_columns = [
    'suspect_age',
    'victim_age',
    'num_arrests',
    'property_loss_usd'
]

outlier_report = []

for col in outlier_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    
    outlier_report.append([
        col,
        Q1,
        Q3,
        lower_bound,
        upper_bound,
        outliers
    ])

outlier_report = pd.DataFrame(
    outlier_report,
    columns=[
        'Column',
        'Q1',
        'Q3',
        'Lower Bound',
        'Upper Bound',
        'Outlier Count'
    ]
)

outlier_report

,Column,Q1,Q3,Lower Bound,Upper Bound,Outlier Count
0,suspect_age,34.00,57.00,-0.500,91.500,284
1,victim_age,30.00,69.00,-28.500,127.500,283
2,num_arrests,1.00,4.00,-3.500,8.500,63
3,property_loss_usd,12643.56,34838.99,-20649.585,68132.135,94


In [20]:
# Cap outliers using IQR method

for col in outlier_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df[col] = df[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Outliers capped using the IQR method.")


Outliers capped using the IQR method.


In [21]:
outlier_check = []

for col in outlier_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    
    outlier_check.append([col, outliers])

outlier_check = pd.DataFrame(
    outlier_check,
    columns=['Column', 'Outliers After Capping']
)

outlier_check


,Column,Outliers After Capping
0,suspect_age,0
1,victim_age,0
2,num_arrests,0
3,property_loss_usd,0


In [22]:
# Correct data types

id_columns = [
    'incident_id',
    'officer_id',
    'suspect_id',
    'victim_id'
]

for col in id_columns:
    df[col] = df[col].astype('string')

# Badge number is an identifier, not a measurement
df['badge_number'] = df['badge_number'].astype('Int64').astype('string')

# Ensure monetary column is float
df['property_loss_usd'] = df['property_loss_usd'].astype(float)


print("Data type correction completed.")
print(df[id_columns + ['badge_number', 'incident_datetime', 'property_loss_usd']].dtypes)


Data type correction completed.
incident_id                  string
officer_id                   string
suspect_id                   string
victim_id                    string
badge_number                 string
incident_datetime    datetime64[us]
property_loss_usd           float64
dtype: object


In [23]:
print("FINAL DATA QUALITY CHECK")
print("=" * 40)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

print("\nRemaining Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nKey Data Types:")
print(df[['incident_datetime', 'property_loss_usd',
          'incident_id', 'suspect_id', 'victim_id']].dtypes)

FINAL DATA QUALITY CHECK
Rows: 4357
Columns: 33
Total Missing Values: 0
Duplicate Rows: 0

Remaining Missing Values:
Series([], dtype: int64)

Key Data Types:
incident_datetime    datetime64[us]
property_loss_usd           float64
incident_id                  string
suspect_id                   string
victim_id                    string
dtype: object


In [24]:
after_rows = len(df)
after_duplicates = df.duplicated().sum()
after_nulls = df.isnull().sum().sum()

before_after = pd.DataFrame({
    'Metric': [
        'Row Count',
        'Duplicate Rows',
        'Total Missing Values'
    ],
    'Before Cleaning': [
        before_rows,
        200,
        16478
    ],
    'After Cleaning': [
        after_rows,
        after_duplicates,
        after_nulls
    ]
})

before_after

,Metric,Before Cleaning,After Cleaning
0,Row Count,5250,4357
1,Duplicate Rows,200,0
2,Total Missing Values,16478,0


In [25]:
output_file = "crime_incidents_cleaned.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully as:", output_file)


Cleaned dataset saved successfully as: crime_incidents_cleaned.csv


## Conclusion

The messy crime incidents dataset was successfully cleaned and transformed into an analysis-ready dataset.


### Key Cleaning Results

- Original dataset contained **5,250 rows and 33 columns**.
- **16,478 missing values** were handled using appropriate strategies.
- **200 duplicate rows** were identified initially, with duplicates remaining after date cleaning removed from the final dataset.
- Invalid and missing incident dates were removed after conversion to datetime.
- Inconsistent text formatting and crime-type labels were standardized.
- Numerical outliers were detected using the **IQR method** and capped to reduce the effect of extreme values.
- IDs, dates, and monetary fields were converted to appropriate data types.
- Final dataset contains **4,357 rows and 33 columns**.
- Final dataset contains **0 missing values and 0 duplicate rows**.

The cleaned dataset is now suitable for further analysis and reporting.